# NB 03 — Clasificación Contable con Memoria
**Proyecto 5 · Automatización Contable**

**Input:** `data/processed/facturas_validadas.json`  
**Output:** `data/processed/facturas_clasificadas.json`  
**Memoria:** `data/memoria/clasificaciones.db` (SQLite)

### Lógica de 3 niveles
1. **RUC nuevo** → Claude clasifica + guarda en memoria
2. **RUC conocido, 1 cuenta** → confianza 0.97, automático
3. **RUC conocido, múltiples cuentas** → score Jaccard de palabras clave; si ≥ 0.60 → confianza 0.85; si no → Claude con contexto de ambigüedad

### Umbrales de revisión
- `≥ 0.90` → auto al Excel
- `0.70–0.89` → Excel + marcado en reporte
- `< 0.70` → cola obligatoria (celda amarilla en Excel)

## 0. Setup

In [1]:
import anthropic
import json
import sqlite3
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import cliente_config

load_dotenv(dotenv_path=Path('D:/Proyecto_Gabriel/02_Agente_IA/Skill_financiero/.env'))

BASE_DIR      = Path('../')
CONFIG_DIR    = BASE_DIR / 'config'
INPUT_PATH    = BASE_DIR / 'data/processed/facturas_extraidas.json'
OUTPUT_PATH   = BASE_DIR / 'data/processed/facturas_clasificadas.json'

# Cliente activo — cambiar este ID para procesar otra empresa/rubro
CLIENTE_ID = 'demo_pitch'
CLIENTE    = cliente_config.cargar_cliente(CLIENTE_ID, CONFIG_DIR)
PLAN_CONTABLE = CLIENTE['plan']
CODIGOS       = CLIENTE['codigos_estructurales']

MEMORIA_PATH  = BASE_DIR / 'data/memoria' / CLIENTE['memoria_db']
MEMORIA_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORTE_PREV  = BASE_DIR / 'data/output' / f'reporte_revision_anterior_{CLIENTE_ID}.xlsx'

# Demo: reiniciar memoria en cada ensayo para partir siempre de cero
RESET_MEMORIA = True
if RESET_MEMORIA and MEMORIA_PATH.exists():
    MEMORIA_PATH.unlink()
    print('⚠️  Memoria reiniciada — clasificaciones.db eliminado')

CUENTAS_GASTO = cliente_config.cuentas_gasto(CLIENTE)

client = anthropic.Anthropic()
api_ok = 'SI' if os.environ.get('ANTHROPIC_API_KEY') else 'NO ENCONTRADA'
print(f'Cliente activo: {CLIENTE["nombre"]} ({CLIENTE["rubro"]})')
print(f'Plan contable cargado: {len(CUENTAS_GASTO)} cuentas de gasto')
print(f'Memoria: {MEMORIA_PATH}')
print(f'API key: {api_ok}')


⚠️  Memoria reiniciada — clasificaciones.db eliminado
Cliente activo: EMPRESA DEMO (retail_mercaderias)
Plan contable cargado: 53 cuentas de gasto
Memoria: ..\data\memoria\clasificaciones_demo_pitch.db
API key: SI


## 1. Motor de Memoria (SQLite)

In [2]:
class MemoriaClasificaciones:
    """
    Almacena y recupera reglas de clasificación contable.
    Patrón: (ruc_proveedor + palabras_clave_descripcion) → cuenta_debe
    """

    def __init__(self, db_path: Path):
        self.db_path = str(db_path)
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS clasificaciones (
                    id                    INTEGER PRIMARY KEY AUTOINCREMENT,
                    ruc_proveedor         TEXT NOT NULL,
                    patron_descripcion    TEXT DEFAULT '',
                    cuenta_debe           TEXT NOT NULL,
                    detalle_cuenta        TEXT,
                    veces_usada           INTEGER DEFAULT 1,
                    ultima_fecha          TEXT,
                    confirmada_por_humano INTEGER DEFAULT 0
                )
            """)
            conn.execute("""
                CREATE TABLE IF NOT EXISTS historial (
                    id                INTEGER PRIMARY KEY AUTOINCREMENT,
                    archivo_origen    TEXT,
                    ruc_proveedor     TEXT,
                    descripcion       TEXT,
                    cuenta_sugerida   TEXT,
                    cuenta_final      TEXT,
                    fue_corregida     INTEGER DEFAULT 0,
                    confianza_sistema REAL,
                    fecha             TEXT
                )
            """)
            conn.commit()

    def buscar_por_ruc(self, ruc: str) -> list[dict]:
        """Retorna todos los patrones conocidos para un RUC, ordenados por uso."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                "SELECT * FROM clasificaciones WHERE ruc_proveedor=? ORDER BY veces_usada DESC, confirmada_por_humano DESC",
                (ruc,)
            ).fetchall()
        return [dict(r) for r in rows]

    def guardar_clasificacion(self, ruc: str, patron: str, cuenta: str, detalle: str):
        """Guarda nueva regla o incrementa veces_usada si ya existe."""
        with sqlite3.connect(self.db_path) as conn:
            existente = conn.execute(
                "SELECT id FROM clasificaciones WHERE ruc_proveedor=? AND cuenta_debe=? AND patron_descripcion=?",
                (ruc, cuenta, patron)
            ).fetchone()
            if existente:
                conn.execute(
                    "UPDATE clasificaciones SET veces_usada=veces_usada+1, ultima_fecha=? WHERE id=?",
                    (datetime.now().isoformat(), existente[0])
                )
            else:
                conn.execute(
                    "INSERT INTO clasificaciones (ruc_proveedor, patron_descripcion, cuenta_debe, detalle_cuenta, ultima_fecha) VALUES (?,?,?,?,?)",
                    (ruc, patron, cuenta, detalle, datetime.now().isoformat())
                )
            conn.commit()

    def confirmar_correccion(self, ruc: str, patron: str, cuenta_final: str, detalle: str):
        """Registra corrección manual de Carlos → confirmada_por_humano=1."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO clasificaciones (ruc_proveedor, patron_descripcion, cuenta_debe, detalle_cuenta, confirmada_por_humano, ultima_fecha) VALUES (?,?,?,?,1,?)",
                (ruc, patron, cuenta_final, detalle, datetime.now().isoformat())
            )
            conn.commit()

    def guardar_historial(self, archivo: str, ruc: str, desc: str, cuenta_sugerida: str, confianza: float):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "INSERT INTO historial (archivo_origen, ruc_proveedor, descripcion, cuenta_sugerida, confianza_sistema, fecha) VALUES (?,?,?,?,?,?)",
                (archivo, ruc, desc, cuenta_sugerida, confianza, datetime.now().isoformat())
            )
            conn.commit()

    def resumen(self) -> dict:
        with sqlite3.connect(self.db_path) as conn:
            n_reglas  = conn.execute("SELECT COUNT(*) FROM clasificaciones").fetchone()[0]
            n_ruc     = conn.execute("SELECT COUNT(DISTINCT ruc_proveedor) FROM clasificaciones").fetchone()[0]
            n_hist    = conn.execute("SELECT COUNT(*) FROM historial").fetchone()[0]
            n_confirm = conn.execute("SELECT COUNT(*) FROM clasificaciones WHERE confirmada_por_humano=1").fetchone()[0]
        return {'reglas': n_reglas, 'rucs_conocidos': n_ruc, 'historial': n_hist, 'confirmadas_humano': n_confirm}


memoria = MemoriaClasificaciones(MEMORIA_PATH)
print('Memoria inicializada:', memoria.resumen())

Memoria inicializada: {'reglas': 0, 'rucs_conocidos': 0, 'historial': 0, 'confirmadas_humano': 0}


## 2. Loop de aprendizaje — leer correcciones anteriores de Carlos

In [3]:
import openpyxl

def cargar_correcciones(reporte_path: Path, memoria: MemoriaClasificaciones):
    """
    Lee el reporte_revision anterior que Carlos llenó con las cuentas finales.
    Columnas (0-indexed): 0=ruc | 2=descripcion | 4=cuenta_sugerida | 9=cuenta_final
    Solo procesa filas donde cuenta_final es distinta a cuenta_sugerida.
    """
    if not reporte_path.exists():
        print('  No hay reporte anterior de correcciones — omitiendo')
        return 0

    wb = openpyxl.load_workbook(str(reporte_path), data_only=True)
    ws = wb.active
    corregidas = 0

    for row in ws.iter_rows(min_row=2, values_only=True):
        ruc          = row[0]
        desc         = row[2]
        cuenta_sug   = row[4]
        cuenta_final = row[9]
        if cuenta_final and str(cuenta_final).strip() != str(cuenta_sug or '').strip():
            detalle = PLAN_CONTABLE['cuentas'].get(str(cuenta_final), '')
            memoria.confirmar_correccion(str(ruc), str(desc or ''), str(cuenta_final), detalle)
            corregidas += 1
            print(f'  Corrección aplicada: RUC {ruc} → {cuenta_sug} → {cuenta_final}')

    return corregidas


print('=== Cargando correcciones anteriores ===')
n = cargar_correcciones(REPORTE_PREV, memoria)
print(f'Total correcciones aplicadas a memoria: {n}')
print('Memoria actualizada:', memoria.resumen())


=== Cargando correcciones anteriores ===
  No hay reporte anterior de correcciones — omitiendo
Total correcciones aplicadas a memoria: 0
Memoria actualizada: {'reglas': 0, 'rucs_conocidos': 0, 'historial': 0, 'confirmadas_humano': 0}


## 3. Motor de clasificación

In [4]:
STOPWORDS = {'de', 'la', 'el', 'los', 'las', 'por', 'para', 'con', 'y', 'en', 'a', 'del',
             'al', 'se', 'su', 'un', 'una', 'es', 'o', 'que', 'e', 'i'}

def score_patron(patron: str, descripcion: str) -> float:
    """Jaccard entre palabras clave del patrón y la descripción de la factura."""
    p = set((patron or '').lower().split()) - STOPWORDS
    d = set((descripcion or '').lower().split()) - STOPWORDS
    if not p:
        return 0.0
    return len(p & d) / len(p)


PROMPT_CLASIFICACION = """
Eres contador peruano experto en PCGE (Plan Contable General Empresarial).
Clasifica esta factura en el plan contable del estudio de contabilidad.

FACTURA:
- RUC emisor: {ruc}
- Razón social: {razon_social}
- Descripción del servicio/producto: {descripcion}
- Monto base imponible: S/ {base_imponible}

CUENTAS DE GASTO DISPONIBLES:
{plan_cuentas}

{contexto_ambiguo}

Devuelve SOLO JSON (sin markdown):
{{
  "cuenta_debe": "6XXXXXX",
  "detalle_cuenta": "nombre completo de la cuenta",
  "patron_para_memoria": "2-4 palabras clave que identifican este tipo de servicio",
  "razon": "una línea explicando la elección de cuenta",
  "confianza": 0.00
}}
"""


def clasificar_con_claude(ruc, razon_social, descripcion, base_imponible, contexto_ambiguo='') -> dict:
    plan_str = '\n'.join(f'  {k}: {v}' for k, v in CUENTAS_GASTO.items())
    prompt = PROMPT_CLASIFICACION.format(
        ruc=ruc,
        razon_social=razon_social or 'No disponible',
        descripcion=descripcion or 'Sin descripción',
        base_imponible=base_imponible or 0,
        plan_cuentas=plan_str,
        contexto_ambiguo=contexto_ambiguo
    )
    resp = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}]
    )
    texto = resp.content[0].text.strip()
    if texto.startswith('```'):
        partes = texto.split('```')
        texto = partes[1][4:] if partes[1].startswith('json') else partes[1]
    return json.loads(texto)


def clasificar(factura: dict, memoria: MemoriaClasificaciones) -> dict:
    """
    Clasifica una factura de COMPRA en 3 niveles:
    1. RUC nuevo → Claude
    2. RUC con 1 cuenta → auto
    3. RUC con múltiples cuentas → keyword match o Claude
    """
    ruc = str(factura.get('ruc_emisor', '') or '').strip()
    razon = factura.get('razon_social_emisor') or factura.get('sunat_razon_social_oficial', '')
    descripcion = factura.get('descripcion_servicio', '')
    base = factura.get('base_imponible', 0)

    registros = memoria.buscar_por_ruc(ruc)

    # NIVEL 1: RUC desconocido
    if not registros:
        resultado_claude = clasificar_con_claude(ruc, razon, descripcion, base)
        confianza = resultado_claude.get('confianza', 0.65)
        cuenta = resultado_claude['cuenta_debe']
        detalle = resultado_claude.get('detalle_cuenta', PLAN_CONTABLE['cuentas'].get(cuenta, ''))
        patron = resultado_claude.get('patron_para_memoria', descripcion[:50])
        memoria.guardar_clasificacion(ruc, patron, cuenta, detalle)
        return {
            'cuenta_debe': cuenta,
            'detalle_cuenta': detalle,
            'confianza_clasificacion': confianza,
            'origen_clasificacion': 'claude_nuevo',
            'razon_clasificacion': resultado_claude.get('razon', ''),
            'requiere_revision': confianza < 0.70
        }

    # NIVEL 2: RUC con una sola cuenta conocida
    cuentas_unicas = set(r['cuenta_debe'] for r in registros)
    if len(cuentas_unicas) == 1:
        r = registros[0]
        memoria.guardar_clasificacion(ruc, r['patron_descripcion'], r['cuenta_debe'], r['detalle_cuenta'] or '')
        return {
            'cuenta_debe': r['cuenta_debe'],
            'detalle_cuenta': r['detalle_cuenta'],
            'confianza_clasificacion': 0.97,
            'origen_clasificacion': 'memoria_exacta',
            'razon_clasificacion': f'RUC visto {r["veces_usada"]} veces, siempre cuenta {r["cuenta_debe"]}',
            'requiere_revision': False
        }

    # NIVEL 3: RUC con múltiples cuentas → match por patrón
    scores = [(score_patron(r['patron_descripcion'], descripcion), r) for r in registros]
    mejor_score, mejor_reg = max(scores, key=lambda x: x[0])

    if mejor_score >= 0.60:
        memoria.guardar_clasificacion(ruc, mejor_reg['patron_descripcion'], mejor_reg['cuenta_debe'], mejor_reg['detalle_cuenta'] or '')
        return {
            'cuenta_debe': mejor_reg['cuenta_debe'],
            'detalle_cuenta': mejor_reg['detalle_cuenta'],
            'confianza_clasificacion': 0.85,
            'origen_clasificacion': 'memoria_patron',
            'razon_clasificacion': f'Score {mejor_score:.2f} con patrón "{mejor_reg["patron_descripcion"]}"',
            'requiere_revision': False
        }

    # Score bajo con múltiples cuentas → Claude con contexto de ambigüedad
    contexto = 'ATENCIÓN — Este RUC tiene clasificaciones previas en múltiples cuentas:\n'
    for r in registros:
        contexto += f'  - Cuenta {r["cuenta_debe"]} ({r["veces_usada"]}x) patrón: "{r["patron_descripcion"]}"\n'
    contexto += 'Elige la más apropiada para ESTA factura según su descripción específica.'

    resultado_claude = clasificar_con_claude(ruc, razon, descripcion, base, contexto_ambiguo=contexto)
    confianza = resultado_claude.get('confianza', 0.70)
    cuenta = resultado_claude['cuenta_debe']
    detalle = resultado_claude.get('detalle_cuenta', PLAN_CONTABLE['cuentas'].get(cuenta, ''))
    patron = resultado_claude.get('patron_para_memoria', descripcion[:50])
    memoria.guardar_clasificacion(ruc, patron, cuenta, detalle)
    return {
        'cuenta_debe': cuenta,
        'detalle_cuenta': detalle,
        'confianza_clasificacion': confianza,
        'origen_clasificacion': 'claude_ambiguo',
        'razon_clasificacion': resultado_claude.get('razon', ''),
        'requiere_revision': confianza < 0.70
    }

print('Motor de clasificación listo')

Motor de clasificación listo


## 4. Procesamiento en lote

In [5]:
TIPOS_AJUSTE = {'NOTA_CREDITO', 'NOTA_DEBITO'}
UMBRAL_BANCARIZACION_SOLES = 2000.0
UMBRAL_BANCARIZACION_USD   = 500.0


def agregar_flags_tributarios(factura: dict) -> None:
    """
    Agrega flags tributarios a la factura según reglas peruanas.
    Bancarización: D.Leg. 1529 / Ley 28194.
    """
    total   = float(factura.get('total') or 0)
    moneda  = factura.get('moneda', 'PEN')
    cod     = factura.get('codigo_tipo_doc', '01')

    # Bancarización
    umbral = UMBRAL_BANCARIZACION_USD if moneda == 'USD' else UMBRAL_BANCARIZACION_SOLES
    factura['requiere_bancarizacion'] = total >= umbral

    # IGV crédito fiscal (si Claude no lo extrajo, inferir desde codigo_tipo_doc)
    if 'otorga_credito_igv' not in factura:
        factura['otorga_credito_igv'] = cod in ('01', '04')

    # Deducibilidad renta (si Claude no lo extrajo, inferir)
    if 'deducible_renta' not in factura:
        factura['deducible_renta'] = cod in ('01', 'R1', '04')


with open(INPUT_PATH, encoding='utf-8') as f:
    facturas = json.load(f)

compras = [f for f in facturas if f.get('tipo_operacion') == 'COMPRA']
ventas  = [f for f in facturas if f.get('tipo_operacion') == 'VENTA']
print(f'A clasificar: {len(compras)} compras, {len(ventas)} ventas')
print('(Las ventas no necesitan clasificación de cuenta gasto — se usa 12121/40111/7041 fijo)')

print('\n=== CLASIFICANDO COMPRAS ===')
for factura in compras:
    archivo   = factura.get('archivo_origen', '?')
    tipo_doc  = factura.get('tipo_doc', 'FACTURA')
    print(f'  {archivo}... ', end='')

    if tipo_doc in TIPOS_AJUSTE:
        # NC/ND: no necesitan clasificación de cuenta gasto — marcar para revisión obligatoria
        factura.update({
            'cuenta_debe':              None,
            'detalle_cuenta':           None,
            'confianza_clasificacion':  1.0,
            'origen_clasificacion':     'ajuste_documental',
            'requiere_revision':        True,
            'razon_clasificacion':      f"{tipo_doc} ref: {factura.get('doc_referencia', 'sin referencia')}"
        })
        agregar_flags_tributarios(factura)
        print(f'⚠️  {tipo_doc} — asiento inverso en NB04 (ref: {factura.get("doc_referencia", "?")})')
        continue

    clasificacion = clasificar(factura, memoria)
    factura.update(clasificacion)
    agregar_flags_tributarios(factura)

    memoria.guardar_historial(
        archivo,
        str(factura.get('ruc_emisor', '')),
        factura.get('descripcion_servicio', ''),
        clasificacion['cuenta_debe'],
        clasificacion['confianza_clasificacion']
    )

    icon = '✅' if clasificacion['confianza_clasificacion'] >= 0.90 else ('⚠️' if clasificacion['confianza_clasificacion'] >= 0.70 else '🔴')
    print(f"{icon} {clasificacion['cuenta_debe']} ({clasificacion['origen_clasificacion']}, conf={clasificacion['confianza_clasificacion']:.2f})")

# Las ventas tienen cuentas fijas — marcar para que NB04 las procese correctamente
print('\n=== CLASIFICANDO VENTAS ===')
for factura in ventas:
    tipo_doc = factura.get('tipo_doc', 'FACTURA')

    if tipo_doc in TIPOS_AJUSTE:
        factura.update({
            'cuenta_debe':             None,
            'confianza_clasificacion': 1.0,
            'origen_clasificacion':    'ajuste_documental',
            'requiere_revision':       True,
            'razon_clasificacion':     f"{tipo_doc} ref: {factura.get('doc_referencia', 'sin referencia')}"
        })
        agregar_flags_tributarios(factura)
        print(f'  {factura.get("archivo_origen","?")} → ⚠️  {tipo_doc} (ref: {factura.get("doc_referencia","?")})')
    else:
        factura['cuenta_debe']             = '12121'
        factura['confianza_clasificacion'] = 1.0
        factura['origen_clasificacion']    = 'fijo_ventas'
        factura['requiere_revision']       = False
        agregar_flags_tributarios(factura)
        print(f'  {factura.get("archivo_origen","?")} → ✅ fijo (12121/40111/7041)')

print(f'\nMemoria actualizada: {memoria.resumen()}')

A clasificar: 1 compras, 5 ventas
(Las ventas no necesitan clasificación de cuenta gasto — se usa 12121/40111/7041 fijo)

=== CLASIFICANDO COMPRAS ===
  WhatsApp Image 2026-07-07 at 15.10.52.jpeg... 

✅ 60110001 (claude_nuevo, conf=0.92)

=== CLASIFICANDO VENTAS ===
  PDF-DOC-E001-52020563066505.pdf → ✅ fijo (12121/40111/7041)
  PDF-DOC-E001-52120563066505.pdf → ✅ fijo (12121/40111/7041)
  PDF-DOC-E001-52220563066505.pdf → ✅ fijo (12121/40111/7041)
  PDF-DOC-E001210771415666.pdf → ✅ fijo (12121/40111/7041)
  PDF-DOC-E001910771415666.pdf → ✅ fijo (12121/40111/7041)

Memoria actualizada: {'reglas': 1, 'rucs_conocidos': 1, 'historial': 1, 'confirmadas_humano': 0}


## 5. Guardar y revisar

In [6]:
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(facturas, f, ensure_ascii=False, indent=2)

print(f'Guardado: {OUTPUT_PATH}')

Guardado: ..\data\processed\facturas_clasificadas.json


In [7]:
import pandas as pd

df = pd.DataFrame(facturas)

print('=== RESUMEN CLASIFICACIÓN ===')
if 'confianza_clasificacion' in df.columns:
    bins = pd.cut(df['confianza_clasificacion'],
                  bins=[0, 0.70, 0.90, 1.01],
                  labels=['🔴 Revisión obligatoria (<0.70)', '⚠️ Revisar (0.70-0.89)', '✅ Auto (≥0.90)'])
    print(bins.value_counts().sort_index())

print('\n=== CLASIFICACIONES POR ORIGEN ===')
if 'origen_clasificacion' in df.columns:
    print(df['origen_clasificacion'].value_counts())

print('\n=== DETALLE COMPRAS ===')
cols = ['archivo_origen', 'razon_social_emisor', 'cuenta_debe', 'confianza_clasificacion', 'razon_clasificacion']
cols_disp = [c for c in cols if c in df.columns]
compras_df = df[df['tipo_operacion']=='COMPRA'][cols_disp]
print(compras_df.to_string(index=False))

=== RESUMEN CLASIFICACIÓN ===
confianza_clasificacion
🔴 Revisión obligatoria (<0.70)    0
⚠️ Revisar (0.70-0.89)            0
✅ Auto (≥0.90)                    6
Name: count, dtype: int64

=== CLASIFICACIONES POR ORIGEN ===
origen_clasificacion
fijo_ventas     5
claude_nuevo    1
Name: count, dtype: int64

=== DETALLE COMPRAS ===
                            archivo_origen              razon_social_emisor cuenta_debe  confianza_clasificacion                                                                                                                             razon_clasificacion
WhatsApp Image 2026-07-07 at 15.10.52.jpeg INVERSIONES VEGA JAUREGUI S.A.C.    60110001                     0.92 Las bebidas alcohólicas y no alcohólicas son mercaderías de consumo clasificadas como abarrotes, registrándose en compra de mercaderías Línea A


In [8]:
# Inspeccionar estado de la memoria
import pandas as pd

with sqlite3.connect(str(MEMORIA_PATH)) as conn:
    df_mem = pd.read_sql('SELECT ruc_proveedor, patron_descripcion, cuenta_debe, veces_usada, confirmada_por_humano FROM clasificaciones ORDER BY veces_usada DESC', conn)

print(f'=== MEMORIA: {len(df_mem)} reglas ===')
print(df_mem.to_string(index=False))

=== MEMORIA: 1 reglas ===
ruc_proveedor          patron_descripcion cuenta_debe  veces_usada  confirmada_por_humano
  20611104968 Bebidas, Abarrotes, Consumo    60110001            1                      0
